In [ ]:
import heapq

class Solution(object):
    def maxScoreWithCostLimit(self, grid, k):
        """
        :type grid: List[List[int]]
        :type k: int
        :rtype: int
        """
        m, n = len(grid), len(grid[0])

        # store midway input
        quantelis = [row[:] for row in grid]

        # helper functions
        def cell_cost(val):
            return 1 if val in [1, 2] else 0
        
        def cell_score(val):
            return val
        
        # Priority queue: (-score, cost, i, j)
        # Negative score → max-heap behavior
        heap = []
        start_cost = cell_cost(grid[0][0])
        start_score = cell_score(grid[0][0])
        if start_cost > k:
            return -1
        
        heapq.heappush(heap, (-start_score, start_cost, 0, 0))
        
        # visited[(i, j, cost)] = max_score
        visited = {}
        visited[(0, 0, start_cost)] = start_score

        max_score = -1
        directions = [(1, 0), (0, 1)]  # down, right

        while heap:
            neg_score, cost, x, y = heapq.heappop(heap)
            score = -neg_score

            # if reached the end within cost limit
            if x == m - 1 and y == n - 1 and cost <= k:
                max_score = max(max_score, score)

            for dx, dy in directions:
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n:
                    new_cost = cost + cell_cost(grid[nx][ny])
                    new_score = score + cell_score(grid[nx][ny])
                    if new_cost <= k and visited.get((nx, ny, new_cost), -1) < new_score:
                        visited[(nx, ny, new_cost)] = new_score
                        heapq.heappush(heap, (-new_score, new_cost, nx, ny))
        
        return max_score if max_score != -1 else -1


# Using DP

In [ ]:
class Solution(object):
    def maxScoreWithCostLimit(self, grid, k):
        """
        :type grid: List[List[int]]
        :type k: int
        :rtype: int
        """
        m, n = len(grid), len(grid[0])

        # store midway input (for debugging or tracking)
        quantelis = [row[:] for row in grid]  # deep copy of the grid

        # initialize dp with -inf
        dp = [[[-float('inf')] * (k + 1) for _ in range(n)] for _ in range(m)]

        # initial cell
        cost = 1 if grid[0][0] in [1, 2] else 0
        score = grid[0][0]
        if cost <= k:
            dp[0][0][cost] = score

        # fill DP
        for i in range(m):
            for j in range(n):
                cell_score = grid[i][j]
                cell_cost = 1 if grid[i][j] in [1, 2] else 0
                for c in range(k + 1):
                    if dp[i][j][c] == -float('inf'):
                        continue
                    # move down
                    if i + 1 < m:
                        new_cost = c + (1 if grid[i+1][j] in [1, 2] else 0)
                        if new_cost <= k:
                            new_score = dp[i][j][c] + grid[i+1][j]
                            dp[i+1][j][new_cost] = max(dp[i+1][j][new_cost], new_score)
                    # move right
                    if j + 1 < n:
                        new_cost = c + (1 if grid[i][j+1] in [1, 2] else 0)
                        if new_cost <= k:
                            new_score = dp[i][j][c] + grid[i][j+1]
                            dp[i][j+1][new_cost] = max(dp[i][j+1][new_cost], new_score)

        # find max score at destination with cost ≤ k
        best = max(dp[m-1][n-1])
        return best if best != -float('inf') else -1
